# Analysis — `n1m5_T15_obs5000_seed42`

Compare the **credulous** vs **vigilant** listener's expected θ trajectory under two speakers (informative, persuasive), across all 9 true thetas.

Figure: **2 rows × 9 columns**.
- Row 1: speaker is `inf` (informative).
- Row 2: speaker is `persp` (persuasive, `pers+`).
- Column k: true θ = 0.1·k.
- Each subplot: x = round (0..15), y = E[θ | u_{0..t}]. Two lines + 95% CI shading: blue = credulous, orange = vigilant. Round 0 is the (flat) prior, recorded round t=0 is plotted as round 1, recorded round t=14 is plotted as round 15.

The notebook is factorized: **Load → Compute → Aggregate → Visualize**, one section each.

## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
from pathlib import Path

# Bootstrap repo root onto sys.path so absolute imports work from a notebook.
HERE = Path.cwd().resolve()
# analyze.ipynb -> n1m5.../ -> simulation_experiments/ -> simulations/ -> models/ -> repo root
REPO_ROOT = HERE.parents[3]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from models.simulations.simulation_experiments.n1m5_T15_obs5000_seed42.io import load_beliefs

DATA_ROOT = HERE / 'raw_do_not_track'
print('DATA_ROOT =', DATA_ROOT)
assert DATA_ROOT.is_dir(), f'expected experiment data at {DATA_ROOT}'

In [ ]:
# Identifiers — must match the directory names produced by run.py.
SPEAKERS = {
    'inf':   'inf_L1strat_a3_b1_uiF',
    'persp': 'persp_L1strat_a3_b0_uiF',
}
LISTENERS = {
    'credulous': 'credulous_L1coop_a3_uiF',
    'vigilant':  'vigilant_L1strat_a3_uiF',
}

THETAS = [round(0.1 * k, 1) for k in range(1, 10)]   # [0.1, ..., 0.9]

# Plot config
LISTENER_COLORS = {'credulous': 'tab:blue', 'vigilant': 'tab:orange'}
SPEAKER_TITLES = {'inf': 'Informative speaker', 'persp': 'Persuasive (pers+) speaker'}

## 1. Load — read belief Datasets for the (speaker × listener) cells we care about

In [ ]:
def load_belief_grid(data_root, speakers, listeners):
    """Return a dict {(spk_key, lst_key): xr.Dataset} for every (speaker, listener) pair."""
    out = {}
    for spk_key, spk_dir in speakers.items():
        for lst_key, lst_dir in listeners.items():
            path = data_root / spk_dir / lst_dir
            out[(spk_key, lst_key)] = load_beliefs(path)
    return out

beliefs = load_belief_grid(DATA_ROOT, SPEAKERS, LISTENERS)
for k, ds in beliefs.items():
    print(f'{k}: sizes={dict(ds.sizes)} path={ds.attrs["execution_path"]}')

## 2. Compute — expected θ per trajectory, with the flat prior prepended at round 0

For each (theta_true, obs_idx, utt_idx, t), compute

$$\mathbb{E}[\theta \mid u_{0..t}] = \sum_\theta \theta \cdot P(\theta \mid u_{0..t}).$$

Then prepend a synthetic round 0 whose value is the prior expectation, $\mathbb{E}_\text{prior}[\theta]$ — for a uniform prior over θ ∈ {0.0, 0.1, …, 1.0}, that's 0.5.

We collapse `obs_idx` × `utt_idx` into a single trajectory axis (`traj`) so downstream aggregation is straightforward; with `n_utt_seq=1` for this experiment the two axes carry the same information.

In [ ]:
def expected_theta_with_prior(belief_ds):
    """
    Returns a numpy array of shape (n_theta_true, n_traj, n_rounds) where
      n_rounds = 1 + (recorded T)
    and round 0 is the prior expectation, derived from belief_ds.theta.
    """
    # E[theta | u_{0..t}] for the recorded rounds.
    e = (belief_ds.belief_theta * belief_ds.theta).sum('theta')
    # Stack obs_idx + utt_idx into one trajectory axis, then put dims in (theta_true, traj, t).
    e = e.stack(traj=('obs_idx', 'utt_idx')).transpose('theta_true', 'traj', 't')
    arr = e.values

    # Prior: flat over theta -> E[theta] = mean of theta grid.
    prior_value = float(belief_ds.theta.mean().item())
    n_thetas, n_traj, T = arr.shape
    prior_block = np.full((n_thetas, n_traj, 1), prior_value, dtype=arr.dtype)
    return np.concatenate([prior_block, arr], axis=2)   # shape (n_thetas, n_traj, T+1)

e_theta = {k: expected_theta_with_prior(ds) for k, ds in beliefs.items()}
for k, arr in e_theta.items():
    print(f'{k}: shape={arr.shape}  (theta_true, traj, round)  range=[{arr.min():.3f}, {arr.max():.3f}]')

## 3. Aggregate — median and 95% interval across trajectories

Compute the median and the 2.5 / 97.5 percentiles across the `traj` axis. Each summary array has shape (n_theta_true, n_rounds).

In [ ]:
def summarize_traj(arr, lo_pct=2.5, hi_pct=97.5):
    """arr: (theta_true, traj, round). Returns (median, lo, hi), each (theta_true, round)."""
    median = np.median(arr, axis=1)
    lo = np.percentile(arr, lo_pct, axis=1)
    hi = np.percentile(arr, hi_pct, axis=1)
    return median, lo, hi

summaries = {k: summarize_traj(arr) for k, arr in e_theta.items()}
# Sanity check on a representative cell
med, lo, hi = summaries[('persp', 'vigilant')]
print(f'persp -> vigilant summary shape: median={med.shape}, lo={lo.shape}, hi={hi.shape}')
print(f"At theta_true=0.5 (idx 4), final round: median={med[4, -1]:.3f}  95%CI=[{lo[4, -1]:.3f}, {hi[4, -1]:.3f}]")

## 4. Visualize

In [ ]:
def plot_belief_grid(summaries, thetas, speakers, listeners,
                     listener_colors, speaker_titles):
    n_rows = len(speakers)
    n_cols = len(thetas)
    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(2.0 * n_cols, 2.5 * n_rows),
        sharex=True, sharey=True,
        squeeze=False,
    )

    # Number of rounds inferred from one summary (first speaker / first listener).
    any_med = next(iter(summaries.values()))[0]
    n_rounds = any_med.shape[1]
    rounds = np.arange(n_rounds)

    for row, spk_key in enumerate(speakers):
        for col, theta_true in enumerate(thetas):
            ax = axes[row, col]
            ax.axhline(theta_true, color='black', linestyle='--',
                       linewidth=0.8, alpha=0.6)   # truth line

            for lst_key in listeners:
                med, lo, hi = summaries[(spk_key, lst_key)]
                color = listener_colors[lst_key]
                ax.plot(rounds, med[col], color=color, label=lst_key, linewidth=1.6)
                ax.fill_between(rounds, lo[col], hi[col], color=color, alpha=0.18)

            if row == 0:
                ax.set_title(f'$\\theta_{{true}}={theta_true}$', fontsize=10)
            if col == 0:
                ax.set_ylabel(f'{speaker_titles[spk_key]}\n$\\mathbb{{E}}[\\theta \\mid u_{{0..t}}]$',
                              fontsize=9)
            if row == n_rows - 1:
                ax.set_xlabel('round')
            ax.set_ylim(0, 1)
            ax.set_xlim(0, n_rounds - 1)

    # Single legend, drawn outside the grid for cleanliness.
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center',
               ncol=len(labels), bbox_to_anchor=(0.5, 1.02), fontsize=10)
    fig.tight_layout(rect=(0, 0, 1, 0.97))
    return fig

fig = plot_belief_grid(
    summaries, THETAS,
    speakers=list(SPEAKERS.keys()),
    listeners=list(LISTENERS.keys()),
    listener_colors=LISTENER_COLORS,
    speaker_titles=SPEAKER_TITLES,
)
plt.show()

## What to look for

- **Truth line** (black dashed) marks $\theta_{true}$ for each column.
- Under the **informative speaker** (row 1), both listeners should track the truth — the credulous listener is *correctly* assuming the speaker is informative.
- Under the **persuasive speaker** (row 2), the credulous listener should be **biased**: it still assumes informativeness, so persuasive utterances move its expected θ in the speaker's preferred direction (`pers+` shifts it up). The vigilant listener's blue-vs-orange separation is the magnitude of the persuasion effect that vigilance corrects for.
- The **CI bands** widen with round only if individual trajectories diverge; they narrow when most trajectories agree on the posterior.